# 5-Year Pipeline: Part 2 - Feature Generation

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calc_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    return true_range.rolling(period).mean()

def calculate_sector_metrics(raw_dir):
    files = glob.glob(os.path.join(raw_dir, '*_data.csv'))
    sector_data = []
    for f in files:
        try:
            df = pd.read_csv(f, usecols=['Date', 'Sector', 'Close'])
            df['Date'] = pd.to_datetime(df['Date'])
            sector_data.append(df)
        except:
            pass
    if not sector_data: return None
    
    all_data = pd.concat(sector_data)
    sector_mean = all_data.groupby(['Date', 'Sector'])['Close'].mean().reset_index()
    sector_pivot = sector_mean.pivot(index='Date', columns='Sector', values='Close')
    return sector_pivot

In [3]:
def calculate_features(df, sector_pivot=None):
    import numpy as np
    df = df.copy()
    
    # 1. Moving Averages
    mas = [15, 30, 50, 100, 150, 200]
    for ma in mas:
        df[f'MA{ma}'] = df['Close'].rolling(window=ma).mean()
        df[f'MA{ma}_slope'] = df[f'MA{ma}'].diff(1) / df[f'MA{ma}'].shift(1) * 100
        df[f'Close_vs_MA{ma}'] = df['Close'] / df[f'MA{ma}']
        
    df['MA15_vs_MA30'] = df['MA15'] / df['MA30']
    df['MA30_vs_MA50'] = df['MA30'] / df['MA50']
    df['MA50_vs_MA100'] = df['MA50'] / df['MA100']
    df['MA50_vs_MA150'] = df['MA50'] / df['MA150']
    df['MA100_vs_MA200'] = df['MA100'] / df['MA200']
    
    # 2. Bollinger Bands
    df['BB_Middle'] = df['Close'].rolling(window=20).mean()
    bb_std = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = df['BB_Middle'] + (bb_std * 2)
    df['BB_Lower'] = df['BB_Middle'] - (bb_std * 2)
    
    df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']
    df['BB_Width_pct'] = df['BB_Width'] / df['BB_Middle']
    df['BB_pctB'] = (df['Close'] - df['BB_Lower']) / df['BB_Width']
    
    df['Close_dist_BB_Upper'] = df['Close'] - df['BB_Upper']
    df['Close_dist_BB_Middle'] = df['Close'] - df['BB_Middle']
    df['Close_dist_BB_Lower'] = df['Close'] - df['BB_Lower']
    df['BB_Width_slope'] = df['BB_Width'].diff()
    
    # 3. RSI / Momentum
    for p in [7, 14, 21]:
        df[f'RSI{p}'] = calc_rsi(df['Close'], period=p)
        
    df['RSI_slope'] = df['RSI14'].diff()
    df['RSI_1D_change'] = df['RSI14'].diff(1)
    df['RSI_3D_change'] = df['RSI14'].diff(3)
    df['RSI_5D_change'] = df['RSI14'].diff(5)
    
    df['RSI_lt_30'] = df['RSI14'] < 30
    df['RSI_lt_40'] = df['RSI14'] < 40
    df['RSI_gt_50'] = df['RSI14'] > 50
    df['RSI_gt_70'] = df['RSI14'] > 70
    
    df['RSI_cross_30'] = (df['RSI14'].shift(1) < 30) & (df['RSI14'] >= 30)
    df['RSI_cross_50'] = (df['RSI14'].shift(1) < 50) & (df['RSI14'] >= 50)
    df['RSI_cross_70'] = (df['RSI14'].shift(1) < 70) & (df['RSI14'] >= 70)
    
    # 4. Price Returns
    for d in [1, 3, 5, 7, 10, 20, 50]:
        df[f'Ret_{d}D'] = df['Close'].pct_change(d) * 100
        
    # 5. Recent Price Structure
    for d in [5, 10, 20, 50, 252]:
        label = f'{d}D' if d != 252 else '52W'
        df[f'{label}_High'] = df['High'].rolling(d).max()
        df[f'{label}_Low'] = df['Low'].rolling(d).min()
        df[f'Dist_{label}_High'] = df['Close'] / df[f'{label}_High'] - 1
        df[f'Dist_{label}_Low'] = df['Close'] / df[f'{label}_Low'] - 1
        
    # 6. Candle Features
    df['Candle_body'] = np.abs(df['Close'] - df['Open'])
    df['Candle_range'] = df['High'] - df['Low']
    df['Upper_wick'] = df['High'] - df[['Open', 'Close']].max(axis=1)
    df['Lower_wick'] = df[['Open', 'Close']].min(axis=1) - df['Low']
    
    cr_safe = df['Candle_range'].replace(0, np.nan)
    df['Body_pct'] = df['Candle_body'] / cr_safe
    df['Upper_wick_pct'] = df['Upper_wick'] / cr_safe
    df['Lower_wick_pct'] = df['Lower_wick'] / cr_safe
    df['Close_pos'] = (df['Close'] - df['Low']) / cr_safe
    
    df['Gap_pct'] = (df['Open'] - df['Close'].shift(1)) / df['Close'].shift(1) * 100
    df['Bullish'] = df['Close'] > df['Open']
    df['Bearish'] = df['Close'] < df['Open']
    
    df['Doji'] = df['Body_pct'] < 0.1
    df['Hammer'] = (df['Lower_wick_pct'] > 0.6) & (df['Upper_wick_pct'] < 0.1) & (df['Body_pct'] < 0.3)
    df['Shooting_Star'] = (df['Upper_wick_pct'] > 0.6) & (df['Lower_wick_pct'] < 0.1) & (df['Body_pct'] < 0.3)
    
    prev_body = df['Candle_body'].shift(1)
    df['Engulfing'] = (df['Candle_body'] > prev_body) & \
                      ((df['Bullish'] & df['Bearish'].shift(1) & (df['Close'] > df['Open'].shift(1)) & (df['Open'] < df['Close'].shift(1))) | \
                       (df['Bearish'] & df['Bullish'].shift(1) & (df['Close'] < df['Open'].shift(1)) & (df['Open'] > df['Close'].shift(1))))
                       
    df['Inside_candle'] = (df['High'] < df['High'].shift(1)) & (df['Low'] > df['Low'].shift(1))
    df['Outside_candle'] = (df['High'] > df['High'].shift(1)) & (df['Low'] < df['Low'].shift(1))
    
    # 7. Volume
    for d in [5, 20, 50]:
        df[f'Vol_{d}D_avg'] = df['Volume'].rolling(d).mean()
        
    df['Vol_vs_20D_avg'] = df['Volume'] / df['Vol_20D_avg'].replace(0, np.nan)
    df['Vol_vs_50D_avg'] = df['Volume'] / df['Vol_50D_avg'].replace(0, np.nan)
    df['Vol_1D_change'] = df['Volume'].pct_change(1)
    df['Vol_5D_change'] = df['Volume'].pct_change(5)
    df['Vol_spike'] = df['Volume'] > df['Vol_20D_avg'] * 2
    
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    df['OBV_slope'] = df['OBV'].diff()
    
    # 8. Volatility
    df['ATR14'] = calc_atr(df, 14)
    df['ATR20'] = calc_atr(df, 20)
    df['ATR_pct'] = df['ATR14'] / df['Close'] * 100
    
    for d in [5, 10, 20, 50]:
        df[f'Volat_{d}D'] = df['Ret_1D'].rolling(d).std()
        
    df['Volat_change'] = df['Volat_10D'].diff()
    
    # 9. Trend Structure
    df['Trend_15_30'] = df['MA15'] > df['MA30']
    df['Trend_30_50'] = df['MA30'] > df['MA50']
    df['Trend_50_100'] = df['MA50'] > df['MA100']
    df['Trend_50_150'] = df['MA50'] > df['MA150']
    df['Trend_100_200'] = df['MA100'] > df['MA200']
    df['Trend_150_200'] = df['MA150'] > df['MA200']
    
    df['Trend_15_30_50_Up'] = df['Trend_15_30'] & df['Trend_30_50']
    df['Trend_15_30_50_Dn'] = (df['MA15'] < df['MA30']) & (df['MA30'] < df['MA50'])
    
    df['Bullish_MA_count'] = df[['Trend_15_30', 'Trend_30_50', 'Trend_50_100', 'Trend_100_200']].sum(axis=1)
    df['MA15_cross_MA50'] = (df['MA15'].shift(1) <= df['MA50'].shift(1)) & (df['MA15'] > df['MA50'])
    
    # 10. Support/Resistance
    df['Support_dist'] = df['Dist_20D_Low']
    df['Resistance_dist'] = df['Dist_20D_High']
    df['Support_touches'] = (df['Low'] <= (df['20D_Low'] * 1.01)).rolling(20).sum()
    df['Resistance_touches'] = (df['High'] >= (df['20D_High'] * 0.99)).rolling(20).sum()
    
    # 11. Market Context
    if 'Nifty50_Close' in df.columns:
        for d in [1, 5, 7, 20]:
            df[f'Nifty_{d}D_ret'] = df['Nifty50_Close'].pct_change(d) * 100
        
        df['Nifty_MA50'] = df['Nifty50_Close'].rolling(50).mean()
        df['Nifty_MA200'] = df['Nifty50_Close'].rolling(200).mean()
        df['Nifty_vs_MA50'] = df['Nifty50_Close'] / df['Nifty_MA50']
        df['Nifty_vs_MA200'] = df['Nifty50_Close'] / df['Nifty_MA200']
        df['Nifty_RSI'] = calc_rsi(df['Nifty50_Close'])
        df['Nifty_Trend'] = df['Nifty_MA50'] > df['Nifty_MA200']
        
        for d in [5, 7, 20]:
            df[f'Rel_str_Nifty_{d}D'] = df[f'Ret_{d}D'] - df[f'Nifty_{d}D_ret']
            
    symbol_sector = df['Sector'].iloc[-1] if 'Sector' in df.columns else 'Unknown'
    if sector_pivot is not None and symbol_sector in sector_pivot.columns:
        sec_close = sector_pivot[symbol_sector]
        df['Sector_Close'] = sec_close.reindex(df.index)
        for d in [1, 5, 7, 20]:
            df[f'Sector_{d}D_ret'] = df['Sector_Close'].pct_change(d) * 100
            if f'Ret_{d}D' in df.columns:
                df[f'Rel_str_Sector_{d}D'] = df[f'Ret_{d}D'] - df[f'Sector_{d}D_ret']
        
        df['Sector_MA50'] = df['Sector_Close'].rolling(50).mean()
        df['Sector_vs_MA50'] = df['Sector_Close'] / df['Sector_MA50']
        df['Sector_RSI'] = calc_rsi(df['Sector_Close'])
            
    # 12. Base Strategy Logic
    df['MA50_trend_7D_up'] = (df['MA50'] > df['MA50'].shift(7)) & (df['MA50'] > df['MA50'].shift(1))
    
    df['Dist_Low_BB'] = df['Low'] - df['BB_Lower']
    df['BB_near_Low'] = (df['Dist_Low_BB'] >= -10) & (df['Dist_Low_BB'] <= 20)
    
    df['Candle_Green'] = df['Close'] > df['Open']
    df['Dist_Low_MA50'] = df['Low'] - df['MA50']
    
    df['Setup_1'] = (df['Dist_Low_MA50'] >= -10) & (df['Dist_Low_MA50'] <= 30)
    df['Setup_2'] = df['Dist_Low_MA50'] > 30
    df['Setup_3'] = df['Dist_Low_MA50'] < -10
    
    df['Base_Signal'] = df['MA50_trend_7D_up'] & df['BB_near_Low'] & df['Candle_Green'] & (df['Setup_1'] | df['Setup_2'] | df['Setup_3'])
    
    df['Signal_Type'] = 0
    df.loc[df['Base_Signal'] & df['Setup_1'], 'Signal_Type'] = 1
    df.loc[df['Base_Signal'] & df['Setup_2'], 'Signal_Type'] = 2
    df.loc[df['Base_Signal'] & df['Setup_3'], 'Signal_Type'] = 3
    
    df['Entry_Price'] = np.where(df['Base_Signal'], df['High'] + 1, np.nan)
    df['Stop_Loss'] = np.where(df['Base_Signal'], df['Low'] - 1, np.nan)
    df['Risk'] = df['Entry_Price'] - df['Stop_Loss']
    df['Target'] = df['Entry_Price'] + (2 * df['Risk'])
    
    # NEW LOGIC: Chronological OHLC forward simulation
    df['Target_Hit'] = False
    df['SL_Hit'] = False
    df['Outcome'] = 'None'
    df['Days_to_Outcome'] = np.nan
    df['Trade_Triggered'] = False
    df['MFE'] = np.nan
    df['MAE'] = np.nan
    df['Future_Ret_7D'] = df['Close'].shift(-7) / df['Close'] - 1
    
    signal_indices = np.where(df['Base_Signal'])[0]
    
    for i in signal_indices:
        if i + 1 >= len(df):
            df.iloc[i, df.columns.get_loc('Outcome')] = 'Pending'
            continue
            
        buy_price = df['Entry_Price'].iloc[i]
        sl_price = df['Stop_Loss'].iloc[i]
        target = df['Target'].iloc[i]
        
        # Day 1 Trigger Check
        day1 = df.iloc[i+1]
        if day1['High'] < buy_price:
            df.iloc[i, df.columns.get_loc('Outcome')] = 'No_Trigger'
            continue
            
        df.iloc[i, df.columns.get_loc('Trade_Triggered')] = True
        
        resolved = False
        mfe_val = day1['High'] - buy_price if day1['High'] > buy_price else 0
        mae_val = buy_price - day1['Low'] if day1['Low'] < buy_price else 0
        
        # Check Day 1 intra-day dynamics (OHLC ambiguity)
        # If open is above buy, triggered at open. If Low hits SL, SL hit.
        day1_sl_hit = False
        day1_target_hit = False
        
        if day1['Low'] <= sl_price:
            if day1['Open'] > sl_price:
                # Open > SL. Did it hit Buy first?
                if day1['Open'] >= buy_price:
                    # Opened above buy, then dipped to SL
                    day1_sl_hit = True
                else:
                    # Opened between SL and Buy. 
                    if day1['High'] >= buy_price:
                        day1_sl_hit = True # conservative assumption
            else:
                # Opened below SL. Rallied to Buy.
                if day1['Close'] <= sl_price:
                    day1_sl_hit = True
                    
        if day1['High'] >= target and not day1_sl_hit:
            day1_target_hit = True
            
        if day1_sl_hit:
            df.iloc[i, df.columns.get_loc('SL_Hit')] = True
            df.iloc[i, df.columns.get_loc('Outcome')] = 'SL'
            df.iloc[i, df.columns.get_loc('Days_to_Outcome')] = 1
            df.iloc[i, df.columns.get_loc('MFE')] = mfe_val
            df.iloc[i, df.columns.get_loc('MAE')] = mae_val
            continue
        elif day1_target_hit:
            df.iloc[i, df.columns.get_loc('Target_Hit')] = True
            df.iloc[i, df.columns.get_loc('Outcome')] = 'Target'
            df.iloc[i, df.columns.get_loc('Days_to_Outcome')] = 1
            df.iloc[i, df.columns.get_loc('MFE')] = mfe_val
            df.iloc[i, df.columns.get_loc('MAE')] = mae_val
            continue
            
        # Loop Day 2 to 7
        for d in range(2, 8):
            if i + d >= len(df):
                df.iloc[i, df.columns.get_loc('Outcome')] = 'Pending'
                resolved = True
                break
                
            day_d = df.iloc[i+d]
            
            # Update MFE / MAE
            if (day_d['High'] - buy_price) > mfe_val: mfe_val = day_d['High'] - buy_price
            if (buy_price - day_d['Low']) > mae_val: mae_val = buy_price - day_d['Low']
            
            sl_hit_d = day_d['Low'] <= sl_price
            target_hit_d = day_d['High'] >= target
            
            if sl_hit_d and target_hit_d:
                df.iloc[i, df.columns.get_loc('SL_Hit')] = True
                df.iloc[i, df.columns.get_loc('Outcome')] = 'SL'
                df.iloc[i, df.columns.get_loc('Days_to_Outcome')] = d
                resolved = True
                break
            elif sl_hit_d:
                df.iloc[i, df.columns.get_loc('SL_Hit')] = True
                df.iloc[i, df.columns.get_loc('Outcome')] = 'SL'
                df.iloc[i, df.columns.get_loc('Days_to_Outcome')] = d
                resolved = True
                break
            elif target_hit_d:
                df.iloc[i, df.columns.get_loc('Target_Hit')] = True
                df.iloc[i, df.columns.get_loc('Outcome')] = 'Target'
                df.iloc[i, df.columns.get_loc('Days_to_Outcome')] = d
                resolved = True
                break
                
        if not resolved:
            df.iloc[i, df.columns.get_loc('Outcome')] = 'Timeout'
            df.iloc[i, df.columns.get_loc('Days_to_Outcome')] = 7
            
        df.iloc[i, df.columns.get_loc('MFE')] = mfe_val
        df.iloc[i, df.columns.get_loc('MAE')] = mae_val
    
    return df


In [4]:
raw_dir = '5_year_data/raw'
generated_dir = '5_year_data/generated'
os.makedirs(generated_dir, exist_ok=True)

# Pre-calculate sector metrics from all raw files
print("Calculating sector metrics...")
sector_pivot = calculate_sector_metrics(raw_dir)
print("Sector metrics calculated!")

# Get list of files
raw_files = glob.glob(os.path.join(raw_dir, '*_data.csv'))
raw_files = sorted(raw_files)

# Show top 5 rows of one file as an example
if raw_files:
    sample_df = pd.read_csv(raw_files[0])
    sample_df['Date'] = pd.to_datetime(sample_df['Date'])
    sample_df.set_index('Date', inplace=True)
    print(f"Sample data loaded: {os.path.basename(raw_files[0])}")
    display(sample_df.head(5))
else:
    print("No raw data files found in", raw_dir)

Calculating sector metrics...


Sector metrics calculated!
Sample data loaded: 360ONE_data.csv


,Stock Symbol,Open,High,Low,Close,Adj Close,Volume,Sector,Sector Index,Nifty50_Open,Nifty50_High,Nifty50_Low,Nifty50_Close,Nifty50_Volume,Trading_Day_Of_Week
Date,,,,,,,,,,,,,,,
2021-08-16,360ONE,390.325012,390.325012,372.500000,375.912506,336.896729,191644,Financial Services,NIFTY FINANCIAL SERVICES,16518.400391,16589.400391,16480.750000,16563.050781,254000.0,Monday
2021-08-17,360ONE,386.250000,400.000000,375.500000,392.200012,351.493744,646280,Financial Services,NIFTY FINANCIAL SERVICES,16545.250000,16628.550781,16495.400391,16614.599609,297400.0,Tuesday
2021-08-18,360ONE,393.250000,404.112488,386.649994,396.575012,355.414734,395400,Financial Services,NIFTY FINANCIAL SERVICES,16691.949219,16701.849609,16535.849609,16568.849609,251800.0,Wednesday
2021-08-20,360ONE,388.000000,395.000000,373.587494,377.262512,338.106689,273612,Financial Services,NIFTY FINANCIAL SERVICES,16382.500000,16509.550781,16376.049805,16450.500000,350800.0,Friday
2021-08-23,360ONE,377.000000,383.649994,369.037506,371.912506,333.311859,260212,Financial Services,NIFTY FINANCIAL SERVICES,16592.250000,16592.500000,16395.699219,16496.449219,275600.0,Monday


In [5]:
# Process all files
total_files = len(raw_files)
print(f"Processing {total_files} files...")

for i, file_path in enumerate(raw_files, 1):
    file_name = os.path.basename(file_path)
    
    try:
        # Load raw data
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        
        # Calculate features
        df_features = calculate_features(df, sector_pivot)
        
        # Save generated data without modifying the raw data
        out_path = os.path.join(generated_dir, file_name)
        df_features.to_csv(out_path)
        
        if i % 10 == 0 or i == total_files:
            print(f"Processed {i}/{total_files}: {file_name}")
            
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

print("All features generated and saved successfully to 5_year_data/generated/!")

Processing 207 files...


Processed 10/207: AMBUJACEM_data.csv


Processed 20/207: AXISBANK_data.csv


Processed 30/207: BHARTIARTL_data.csv


Processed 40/207: CHOLAFIN_data.csv


...

Processed 170/207: SHREECEM_data.csv


Processed 180/207: TATACOMM_data.csv


Processed 190/207: TMCV_data.csv


Processed 200/207: VBL_data.csv


Processed 207/207: ZYDUSLIFE_data.csv
All features generated and saved successfully to 5_year_data/generated/!
